## Quiet the logs

`scarlets/utils/RedisLogger.py` calls `logging.basicConfig(level=DEBUG)` at
**import time**, which sets the *root* logger to DEBUG for the whole kernel -
so every library that logs (redis-py, httpx, asyncio) starts emitting DEBUG
into your cell output as soon as anything imports `scarlets`.

Run this first. It configures the root logger before `scarlets` gets the
chance, which makes that `basicConfig` call a no-op (`basicConfig` does
nothing when the root logger already has handlers).

If you ever see lines like `Failed to enable maintenance notifications:
unknown subcommand 'MAINT_NOTIFICATIONS'` - those are harmless. redis-py 8.x
probes for a Redis 8 maintenance feature on every new connection, Redis 7.x
replies "unknown subcommand", and the client logs it at DEBUG and carries on
with a perfectly normal connection.

In [1]:
import logging

# Claim the root logger first. logging.basicConfig() does nothing if the
# root logger already has handlers, so configuring it here makes
# RedisLogger's own basicConfig(level=DEBUG) - which runs when scarlets is
# imported further down - a no-op instead of a takeover.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# Belt and braces: if something already imported scarlets in this kernel
# (a re-run, or an earlier notebook), the call above was itself the no-op,
# so set the level explicitly too.
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


# 21 — One-sample z-test for a proportion across the fleet

A one-sample z-test compares an observed proportion (0/1 data) against
a hypothesised proportion p0. The test statistic is:

    z = (p̂ - p0) / sqrt(p0 * (1 - p0) / n)

where p̂ is the sample proportion. The two-sided p-value is reported.

Ground truth is computed directly from the same CSVs at the bottom, so the
fleet's answer is checked rather than admired.


## Parameters

In [2]:
import os, time, json, subprocess, random

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
workers       : ['worker1', 'worker2', 'worker3', 'worker4']
expecting     : ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']
fleet managed : Gustavo


## Generate each worker's matrix data and its local config

Four workers, four different shapes, four different column sets. The
intersection is never written down anywhere — it has to be discovered.

`HOME` for each worker points into this shared volume, so
`~/.scarlet/config.yaml` resolves to a path this notebook can write while
the worker still reads it from its real default location.


In [ ]:
import numpy as np
import pandas as pd
import yaml

DATA_DIR = os.environ.get("DATA_DIR", "/data")

# torque_nm is deliberately held by only two of the four workers. Nothing
# in the paths or the source names says which - the head has to ask.
LAYOUT = {
    "worker1": ("plant_alpha/turbine_vibration.csv",  "turbine_alpha_vibration",  80,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw"]),
    "worker2": ("feeds/grid_telemetry.csv",           "grid_telemetry_feed",     100,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "power_kw"]),
    "worker3": ("archive/2026/q3/unit7_sensors.csv",  "unit7_archive_q3",        120,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw",
                 "ambient_temp_c"]),
    "worker4": ("raw_drop/sensor_export_FINAL.csv",   "bulk_sensor_drop",         90,
                ["vibration_rms", "shaft_speed_rpm", "power_kw"]),
}

PARAMS = {
    "vibration_rms":   (2.5,    0.6),
    "bearing_temp_c":  (65.0,   8.0),
    "shaft_speed_rpm": (1800.0, 120.0),
    "torque_nm":       (340.0,  45.0),
    "power_kw":        (64.0,   11.0),
    "ambient_temp_c":  (22.0,   4.0),
}

for i, (node, (relpath, source_name, rows, cols)) in enumerate(LAYOUT.items(), start=1):
    rng = np.random.default_rng(seed=i * 1000)
    start = pd.Timestamp("2026-01-01T00:00:00")
    frame = {"timestamp": [start + pd.Timedelta(minutes=k) for k in range(rows)]}
    for c in cols:
        mean, sd = PARAMS[c]
        frame[c] = np.clip(rng.normal(mean, sd, size=rows), 0.0, None)
    df = pd.DataFrame(frame)

    gap_cols = list(rng.permutation(cols))[:3]
    gap_rows = rng.choice(rows, size=3, replace=False)
    for r, c in zip(gap_rows, gap_cols):
        df.at[r, c] = np.nan

    path = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)

    scarlet_dir = os.path.join(DATA_DIR, node, ".scarlet")
    os.makedirs(scarlet_dir, exist_ok=True)
    actual_gaps = sorted(df.columns[df.isna().any()].tolist())
    with open(os.path.join(scarlet_dir, "config.yaml"), "w") as fh:
        yaml.safe_dump({"sources": [{
            "name": source_name, "type": "csv", "mode": "local",
            "description": (f"{rows} rows. Columns: {', '.join(cols)}. "
                            f"gaps in columns: {', '.join(actual_gaps)}"),
            "path": path,
        }]}, fh, sort_keys=False)

    has = "has torque_nm" if "torque_nm" in cols else "NO torque_nm"
    print(f"{source_name:26s} {rows:>4} rows  {has}")

# Ground truth. The fleet will agree on the columns every worker holds -
# the intersection - so compute over exactly those.
_sets = [set(v[3]) for v in LAYOUT.values()]
_common = sorted(set.intersection(*_sets))
_parts = [pd.read_csv(os.path.join(DATA_DIR, v[0]))[_common].dropna().values
          for v in LAYOUT.values()]
_all = np.vstack(_parts)
_n = _all.shape[0]
_mean = _all.sum(axis=0) / _n
_var = (_all ** 2).sum(axis=0) / _n - _mean ** 2
print(f"\ncommon columns the fleet must discover: {_common}")
print(f"expected n = {_n}")
for _c, _m, _v in zip(_common, _mean, _var):
    print(f"  {_c:18s} mean={_m:12.4f}  variance={_v:12.4f}")


# Check it

`n` should match the ground truth above. A wrong `n` means the row count
used by the fleet differs from what we computed.


## Bring the fleet up

In [4]:
#now we are going to bring the workers up. these can be made fully decentralized using Gustavo!

AGENT_SERVICES = ["agent-head"] + [f"agent-{n}" for n in WORKER_NODES]

def compose(*args):
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result

if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    missing_llm = [k for k in ("LLM_BASE_URL", "LLM_MODEL")
                   if not os.environ.get(k, "").strip()]
    if missing_llm:
        raise RuntimeError(
            f"{', '.join(missing_llm)} not set - the agent head cannot start "
            f"without them.\nSet these in examples/notebooks/.env and restart "
            f"the stack (docker compose up -d), then re-run this notebook."
        )
    
    print("bringing up:", ", ".join(AGENT_SERVICES))
    #compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

Gustavo owns this fleet - not launching anything.
If the wait below times out, check the Gustavo deployment.


## Wait for the fleet to come online

In [5]:
from scarlets.messaging import Messenger

FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS!r}: "
                f"{sorted(missing)}\nSeen on the bus: {sorted(live_agents())}\n"
                f"Check {where}."
            )
        time.sleep(2)

wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

fleet online: ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']


{'__composer_api__',
 'loop-agent-head_head',
 'loop-agent-worker_worker1',
 'loop-agent-worker_worker2',
 'loop-agent-worker_worker3',
 'loop-agent-worker_worker4',
 'notebook_probe'}

## The window, and what the answer should be

Each worker holds one-minute samples starting `2026-01-01 00:00:00`, but a
different number of them — 80, 100, 120 and 90 rows. So the fleet's
coverage is ragged on purpose:

| window | workers with data |
|---|---|
| `[00:00, 00:30)` | all four |
| `[01:40, 02:00)` | **worker3 only** — the others ran out at 01:19, 01:39 and 01:29 |

Ground truth is computed here with pandas over the same files, dropping
rows exactly the way the harness does: `_convert_to_matrix` discards a row
if **any** selected column fails to convert, so a single gap takes the
whole row with it.

In [6]:
import pandas as pd

# The columns every worker holds. Not written into the prompt - the fleet
# discovers this by consensus. Stated here only to check the answer.
SHARED = ["vibration_rms", "shaft_speed_rpm", "power_kw"]

WINDOW_A = ("2026-01-01T00:00:00", "2026-01-01T00:30:00")   # all four workers
WINDOW_B = ("2026-01-01T01:40:00", "2026-01-01T02:00:00")   # worker3 alone

SOURCES = [p for p, _, _, _ in LAYOUT.values()]

def ground_truth(start, end, columns=SHARED):
    """Pool every worker's rows in [start, end) and reduce, as the fleet should."""
    frames = []
    for rel in SOURCES:
        df = pd.read_csv(f"{DATA_DIR}/{rel}", parse_dates=["timestamp"])
        df = df[(df["timestamp"] >= pd.Timestamp(start)) &
                (df["timestamp"] <  pd.Timestamp(end))]
        # Match _convert_to_matrix: a row with a gap in ANY selected column
        # is dropped whole, not imputed and not partially counted.
        frames.append(df[columns].dropna(how="any"))
    pooled = pd.concat(frames, ignore_index=True)
    return pooled.mean(), pooled.var(ddof=0), len(pooled)

def per_source_counts(start, end, columns=SHARED):
    out = {}
    for name, (rel, src_name, _, _) in LAYOUT.items():
        df = pd.read_csv(f"{DATA_DIR}/{rel}", parse_dates=["timestamp"])
        df = df[(df["timestamp"] >= pd.Timestamp(start)) &
                (df["timestamp"] <  pd.Timestamp(end))]
        out[name] = len(df[columns].dropna(how="any"))
    return out

mean_a, var_a, n_a = ground_truth(*WINDOW_A)
print(f"WINDOW A {WINDOW_A[0]} -> {WINDOW_A[1]}")
print(f"  rows per worker: {per_source_counts(*WINDOW_A)}")
print(f"  n = {n_a}")
for c in SHARED:
    print(f"  {c:<18} mean={mean_a[c]:>12.4f}  var={var_a[c]:>14.4f}")

mean_b, var_b, n_b = ground_truth(*WINDOW_B)
print(f"\nWINDOW B {WINDOW_B[0]} -> {WINDOW_B[1]}")
print(f"  rows per worker: {per_source_counts(*WINDOW_B)}")
print(f"  n = {n_b}")
for c in SHARED:
    print(f"  {c:<18} mean={mean_b[c]:>12.4f}  var={var_b[c]:>14.4f}")

WINDOW A 2026-01-01T00:00:00 -> 2026-01-01T00:30:00
  rows per worker: {'worker1': 29, 'worker2': 29, 'worker3': 30, 'worker4': 29}
  n = 117
  vibration_rms      mean=      2.5652  var=        0.3218
  shaft_speed_rpm    mean=   1805.2485  var=    12617.6704
  power_kw           mean=     65.7613  var=      107.9825

WINDOW B 2026-01-01T01:40:00 -> 2026-01-01T02:00:00
  rows per worker: {'worker1': 0, 'worker2': 0, 'worker3': 20, 'worker4': 0}
  n = 20
  vibration_rms      mean=      2.6388  var=        0.7823
  shaft_speed_rpm    mean=   1799.9898  var=    13388.9662
  power_kw           mean=     64.3645  var=      137.1499


## Question 1 — a window every worker can answer

The prompt names a time range and nothing else. No source, no path, no
column, and no SQL. The head has to turn the range into structured
conditions and pass the same ones to every worker.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


answer_a = await ask_scarlet_agent(
    "Compute the per-column mean of the sensor measurements held across the "
    "fleet, but only for readings timestamped from 2026-01-01T00:00:00 up to "
    "but not including 2026-01-01T00:30:00. Report, for each column: its name "
    "and its mean, and state n once. Do not name any data source or column "
    "yourself and do not write any SQL - the workers know their own data."
)
print(answer_a)

### Check it

`n` is the one to read first. The window covers 30 minutes on four
workers, so a correct `n` is near 120 — minus whole rows lost to gaps (117 here). An
`n` close to **380** would mean the filter never reached the workers and
the fleet quietly answered the unfiltered question.

In [8]:
print(f"expected z = {z_true:.10f}")
print(f"expected p = {p_val_true:.10f}")
print()
print("Compare against the head's answer above. The failure this is looking")
print("for is not a wrong-looking number - it is a believable one computed")
print("over the wrong rows or with the wrong formula.")


expected n = 117   (unfiltered would be 380)

  vibration_rms      mean=      2.5652
  shaft_speed_rpm    mean=   1805.2485
  power_kw           mean=     65.7613

Compare against the head's answer above. The failure this is looking
for is not a wrong-looking number - it is a believable one computed
over the wrong rows.


## Question 2 — a window only one worker can answer

`[01:40, 02:00)` is past the end of three workers' data. They match zero
rows, which is an ordinary outcome, not a failure: they must not abort the
round and must not contribute a zero that drags the mean down.

A correct answer here equals worker3's own mean over that window. A mean
that looks like a quarter of it means three empty workers were averaged in
as zeros.

In [ ]:
answer_b = await ask_scarlet_agent(
    "Compute the per-column mean of the sensor measurements held across the "
    "fleet, but only for data between 1:40 am to 2 am on 1/1/2026. Report, for each column: its name "
    "and its mean, and state n once. If some workers hold no readings in that "
    "range, say which."
)
print(answer_b)

In [10]:
print(f"expected n ≈ {n_b}  (worker3 alone)")
print(f"rows per worker: {per_source_counts(*WINDOW_B)}")
print()
for c in SHARED:
    quarter = mean_b[c] / 4
    print(f"  {c:<18} correct={mean_b[c]:>12.4f}   if-zeros-averaged-in≈{quarter:>12.4f}")

expected n ≈ 20  (worker3 alone)
rows per worker: {'worker1': 0, 'worker2': 0, 'worker3': 20, 'worker4': 0}

  vibration_rms      correct=      2.6388   if-zeros-averaged-in≈      0.6597
  shaft_speed_rpm    correct=   1799.9898   if-zeros-averaged-in≈    449.9974
  power_kw           correct=     64.3645   if-zeros-averaged-in≈     16.0911


## Teardown

In [11]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    #compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")

Gustavo owns this fleet - leaving it alone.
